In [4]:
import pandas as pd
import numpy as np

#carga los datos

df_operacion = pd.read_csv('operacion_centros.csv')
df_geograficos = pd.read_csv('centros_geograficos.csv')

print(" Filas en operacion_centros:", len(df_operacion))
print(" Centros geográficos:", len(df_geograficos))
print("\n Vista previa de coordenadas (aquí verás el error del C04):")
print(df_geograficos[['centro_id','nombre_centro','latitud','longitud']])


# Déficit diario = demanda - capacidad (si sale positivo hay deficis)

df_operacion['deficit_diario'] = df_operacion['demanda_paquetes'] - df_operacion['capacidad_paquetes']

# Agrupar por centro_id y sumar el déficit total
deficit_por_centro = df_operacion.groupby('centro_id')['deficit_diario'].sum().reset_index()
deficit_por_centro.rename(columns={'deficit_diario': 'deficit_total_paquetes'}, inplace=True)

print("\n Déficit acumulado por centro:")
print(deficit_por_centro.sort_values('deficit_total_paquetes', ascending=False))

#asocia datos geograficos

df_unido = pd.merge(df_geograficos, deficit_por_centro, on='centro_id', how='inner')


# convirte cordenadas a numero

df_unido['latitud'] = pd.to_numeric(df_unido['latitud'], errors='coerce')
df_unido['longitud'] = pd.to_numeric(df_unido['longitud'], errors='coerce')

print("\n  Coordenadas ANTES del filtro (observa el C04 invertido):")
print(df_unido[['centro_id','nombre_centro','latitud','longitud']])


# filtro de territorio

# Guatemala: Latitud [13, 15], Longitud [-91, -87]
df_validado = df_unido[
    (df_unido['latitud'] >= 13) & (df_unido['latitud'] <= 15) &
    (df_unido['longitud'] >= -91) & (df_unido['longitud'] <= -87)
].copy()

print(f"\n  Registros válidos: {len(df_validado)} de {len(df_unido)}")
print(" Centros EXCLUIDOS por coordenadas inválidas:")
excluidos = df_unido[~df_unido['centro_id'].isin(df_validado['centro_id'])]
print(excluidos[['centro_id','nombre_centro','latitud','longitud']])


# exporta el  geo_validado.csv

df_exportar = df_validado[['centro_id','latitud','longitud','deficit_total_paquetes']]
df_exportar.to_csv('geo_validado.csv', index=False, encoding='utf-8')

print("\n Archivo exportado: geo_validado.csv")
print(df_exportar)

# Descargar el archivo a tu computadora
from google.colab import files
files.download('geo_validado.csv')

 Filas en operacion_centros: 1620
 Centros geográficos: 9

 Vista previa de coordenadas (aquí verás el error del C04):
  centro_id             nombre_centro  latitud  longitud
0       C01             Hub Santa Ana  13.9942  -89.5597
1       C02               Hub Metapán  14.3333  -89.4500
2       C03              Hub Acajutla  13.5917  -89.8281
3       C04             Hub Soyapango -89.1370   13.7102
4       C05                 Hub Apopa  13.8072  -89.1793
5       C06         Hub Quezaltepeque  13.8333  -89.2833
6       C07            Hub San Miguel  13.4833  -88.1833
7       C08     Hub Santiago de María  13.4833  -88.4667
8       C09  Hub San Francisco Gotera  13.6967  -88.1050

 Déficit acumulado por centro:
  centro_id  deficit_total_paquetes
6       C07                   33972
4       C05                   28688
3       C04                   27815
0       C01                   14575
8       C09                   12294
5       C06                   10137
1       C02                

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
# hallazgo 1 :
# En la simbología por magnitud,
# los dos centros con los circulos mas grandes
# son C07 (Hub San Miguel, Oriente) con 33,972 paquetes y C05 (Hub Apopa, Centro) con 28,688 paquetes.
# Los centros de la región Occidente (C01, C02, C03) presentan los valores más bajos

# Explicación causal 1 :
# La región Centro (Apopa) concentra la mayor densidad poblacional del area de San Salvador, haciendo una gran demanda de estructura mas de lo que puede tener de capacidad instalada.
# El caso de San Miguel (Oriente) es mas critico porque, aunque la densidad es menor, opera como un distribuidor para todo el oriente del país, acumulando demanda de múltiples municipios. En contraste, los centros de Occidente tienen menor
# presión demográfica y mejor balance capacidad y demandas, lo que explica por que su valores son cercanos a 0 o negativo.

In [ ]:
# hallazgo 2 :
# una observacion que pude notar al aplicar el filtro de plausibilidad territorial (latitud entre 13 y 15, longitud entre −91 y −87),
# el registro del centro C04 (Hub Soyapango) fue excluido del mapa porque sus coordenadas originales
# aparecen como latitud = −89.137 y longitud = 13.7102, valores que caen fuera del territorio de El Salvador (como la latitud es negativa y la longitud es positiva, me da un resultado justo al revés de lo esperado).

# Explicación causal 1 :
# noto que este este error puede aver sido una transposición de columnas en el sistema de origen:
# los valores de latitud y longitud fueron intercambiados al exportar los datos .
# de haberse incluido sin validar, el punto habría aparecido en el océano Pacífico sur o en una ubicación que ni existe, dandome un error en el análisis espacial de cobertura, las rutas de redistribución logística.

In [ ]:
# Justificación técnica :
# de los 9 centros registrados en centros_geograficos.csv, 8 recibieron validación territorial correcta y 1 fue excluido (C04 Soyapango) por presentar coordenadas invertidas
# (lat=−89.137, lon=13.7102), fuera del rango geográfico de El Salvador. El centro con el mayor déficit válido es C07 (San Miguel) con 33,972 paquetes acumulados, seguido por C05 (Apopa) con 28,688.
# muestra las coordenadas erroneas en el mapa genera riesgos operativos graves: un ejemplo de esto pueden ser las rutas de redistribucion dirigidas hacia ubicaciones inexistentes, fallos en el análisis de cobertura territorial y decisiones logísticas basadas en datos geográficos falsos,
# lo que puede derivar en sobrecostos de transporte fallas y hasta en una posible atención de una demanda por fallo asi.